In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mdtamzidulislam/heartdisease")

print("Path to dataset files:", path)

100%|██████████| 3.39k/3.39k [00:00<00:00, 2.41MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/mdtamzidulislam/heartdisease/versions/1


In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv(path+'/heart.csv')

# Select only the 4 features we want to use
df = df[['age', 'sex', 'cp', 'thall', 'output']]

# Handle missing values (if any) by filling with the median for numeric columns
df.fillna(df.median(), inplace=True)

# Encode categorical columns (cp: chest pain type, 0-3; sex: male=1, female=0)
df['sex'] = df['sex'].map({1: 1, 0: 0})
df['cp'] = df['cp'].map({0: 0, 1: 1, 2: 2, 3: 3})  # chest pain type

# Split the data into features (X) and target (y)
X = df[['age', 'sex', 'cp', 'thall']]
y = df['output']

# Inspect the first few rows
df.head()


,age,sex,cp,thall,output
0,63,1,3,1,1
1,37,1,2,2,1
2,41,0,1,2,1
3,56,1,1,2,1
4,57,0,0,2,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   age     303 non-null    int64
 1   sex     303 non-null    int64
 2   cp      303 non-null    int64
 3   thall   303 non-null    int64
 4   output  303 non-null    int64
dtypes: int64(5)
memory usage: 12.0 KB


In [ ]:
print("Frequency Table for 'ouput':")
df['output'].value_counts()


Frequency Table for 'ouput':


,count
output,
1,165
0,138


In [ ]:

# Frequency table for the features
print("\nFrequency Table for 'sex':")
df['sex'].value_counts()



Frequency Table for 'sex':


,count
sex,
1,207
0,96


In [ ]:

print("\nFrequency Table for 'cp':")
df['cp'].value_counts()


Frequency Table for 'cp':


,count
cp,
0,143
2,87
1,50
3,23


In [ ]:
print("\nFrequency Table for 'thall':")
df['thall'].value_counts()



Frequency Table for 'thall':


,count
thall,
2,166
3,117
1,18
0,2


In [ ]:
print("\nFrequency Table for 'age':")
df['age'].value_counts()


Frequency Table for 'age':


,count
age,
58,19
57,17
54,16
59,14
52,13
51,12
62,11
60,11
44,11


In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [ ]:

# Calculate the likelihood for each column
for column in df.columns:
  print(f"Likelihood for column '{column}':")
  if pd.api.types.is_numeric_dtype(df[column]):
    # For numeric columns, calculate the probability density function (PDF)
    # You might need to choose an appropriate distribution based on your data
    # Here, we assume a normal distribution for example
    mean = df[column].mean()
    std = df[column].std()
    print(f"Mean: {mean}, Standard Deviation: {std}")  # Print mean and std for reference
    # You can use a library like scipy.stats to calculate PDF if needed
  else:
    # For categorical columns, calculate the relative frequency of each value
    value_counts = df[column].value_counts(normalize=True)
    print(value_counts)
  print("\n")


Likelihood for column 'age':
Mean: 54.366336633663366, Standard Deviation: 9.082100989837858


Likelihood for column 'sex':
Mean: 0.6831683168316832, Standard Deviation: 0.4660108233396251


Likelihood for column 'cp':
Mean: 0.966996699669967, Standard Deviation: 1.0320524894832992


Likelihood for column 'thall':
Mean: 2.3135313531353137, Standard Deviation: 0.6122765072781412


Likelihood for column 'output':
Mean: 0.5445544554455446, Standard Deviation: 0.4988347841643926




In [ ]:
# Split the data into two classes (target = 0 and target = 1)
class_0 = df[df['output'] == 0]
class_1 = df[df['output'] == 1]

# Calculate mean and variance for each feature for target=0
mean_0 = class_0[['age', 'sex', 'cp', 'thall']].mean()
var_0 = class_0[['age', 'sex', 'cp', 'thall']].var()

# Calculate mean and variance for each feature for target=1
mean_1 = class_1[['age', 'sex', 'cp', 'thall']].mean()
var_1 = class_1[['age', 'sex', 'cp', 'thall']].var()

# Print the means and variances for each class
print("\nMean and Variance for Class 0 (output = 0):")
print(mean_0)
print(var_0)
print("\n")


print("\nMean and Variance for Class 1 (output = 1):")
print(mean_1)
print(var_1)



Mean and Variance for Class 0 (output = 0):
age      56.601449
sex       0.826087
cp        0.478261
thall     2.543478
dtype: float64
age      63.394742
sex       0.144716
cp        0.820692
thall     0.468899
dtype: float64



Mean and Variance for Class 1 (output = 1):
age      52.496970
sex       0.563636
cp        1.375758
thall     2.121212
dtype: float64
age      91.214930
sex       0.247450
cp        0.906726
thall     0.216925
dtype: float64


In [ ]:
from math import pi, exp, sqrt

# Function to calculate Gaussian likelihood
def gaussian_likelihood(x, mean, var):
    return (1 / sqrt(2 * pi * var)) * exp(-(x - mean)**2 / (2 * var))

# Example: calculate the likelihood of `age=63` for target=1
age_value = 63
likelihood_age_class_1 = gaussian_likelihood(age_value, mean_1['age'], var_1['age'])

print(f"Likelihood of age=63 for class target=1: {likelihood_age_class_1}")


Likelihood of age=63 for class target=1: 0.022817247489412554


In [ ]:
# Function to calculate Gaussian likelihood
def gaussian_likelihood(x, mean, var):
    if var == 0:  # Handle cases with zero variance
        if x == mean:
            return 1.0
        else:
            return 0.0
    return (1 / sqrt(2 * pi * var)) * exp(-(x - mean)**2 / (2 * var))

# Example: calculate the likelihood of a new data point for each class
new_data_point = {'age': 63, 'sex': 1, 'cp': 3, 'thall': 2}

likelihood_class_0 = 1
likelihood_class_1 = 1

for feature in ['age', 'sex', 'cp', 'thall']:
    likelihood_class_0 *= gaussian_likelihood(new_data_point[feature], mean_0[feature], var_0[feature])
    likelihood_class_1 *= gaussian_likelihood(new_data_point[feature], mean_1[feature], var_1[feature])


print(f"Likelihood of the new data point for class 0: {likelihood_class_0}")
print(f"Likelihood of the new data point for class 1: {likelihood_class_1}")


Likelihood of the new data point for class 0: 0.00013327345887699074
Likelihood of the new data point for class 1: 0.0010086798418631347


### Guassian NB

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, accuracy_score

# Initialize the Gaussian Naive Bayes model
gnb = GaussianNB()

# Train the model
gnb.fit(X_train, y_train)

# Make predictions
y_pred_gnb = gnb.predict(X_test)

# Confusion Matrix
cm_gnb = confusion_matrix(y_test, y_pred_gnb)
print("Gaussian NB Confusion Matrix:\n", cm_gnb)

# Accuracy
accuracy_gnb = accuracy_score(y_test, y_pred_gnb)
print(f'Gaussian NB Accuracy: {accuracy_gnb}')


Gaussian NB Confusion Matrix:
 [[32  9]
 [13 37]]
Gaussian NB Accuracy: 0.7582417582417582


### MultiNomial NB

In [ ]:
# Example: Calculate the likelihood of `cp=1` for class target=1
cp_value = 1
likelihood_cp_class_1 = feature_counts_class_1.loc[cp_value, 'cp'] / feature_counts_class_1['cp'].sum()

print(f"Likelihood of cp=1 for class target=1: {likelihood_cp_class_1}")


Likelihood of cp=1 for class target=1: 0.24848484848484848


In [ ]:
from sklearn.naive_bayes import MultinomialNB

# Multinomial Naive Bayes works best with count data, so let's round the features to integers for this demonstration.
X_train_mnb = X_train.apply(lambda x: x.astype(int))
X_test_mnb = X_test.apply(lambda x: x.astype(int))

# Initialize and train Multinomial Naive Bayes
mnb = MultinomialNB()
mnb.fit(X_train_mnb, y_train)

# Make predictions
y_pred_mnb = mnb.predict(X_test_mnb)

# Confusion Matrix
cm_mnb = confusion_matrix(y_test, y_pred_mnb)
print("Multinomial NB Confusion Matrix:\n", cm_mnb)

# Accuracy
accuracy_mnb = accuracy_score(y_test, y_pred_mnb)
print(f'Multinomial NB Accuracy: {accuracy_mnb}')


Multinomial NB Confusion Matrix:
 [[34  7]
 [13 37]]
Multinomial NB Accuracy: 0.7802197802197802


### Bernouli NB

In [ ]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.preprocessing import Binarizer

# Binarize the features (for demonstration purposes)
binarizer = Binarizer()
X_train_bnb = binarizer.fit_transform(X_train)
X_test_bnb = binarizer.transform(X_test)

# Initialize and train Bernoulli Naive Bayes
bnb = BernoulliNB()
bnb.fit(X_train_bnb, y_train)

# Make predictions
y_pred_bnb = bnb.predict(X_test_bnb)

# Confusion Matrix
cm_bnb = confusion_matrix(y_test, y_pred_bnb)
print("Bernoulli NB Confusion Matrix:\n", cm_bnb)

# Accuracy
accuracy_bnb = accuracy_score(y_test, y_pred_bnb)
print(f'Bernoulli NB Accuracy: {accuracy_bnb}')


Bernoulli NB Confusion Matrix:
 [[34  7]
 [13 37]]
Bernoulli NB Accuracy: 0.7802197802197802
